# Gemini API: Entity extraction

Use Gemini API to speed up some of your tasks, such as searching through text to extract needed information. Entity extraction with a Gemini model is a simple query, and you can ask it to retrieve its answer in the form that you prefer.

This notebook shows how to extract entities into a list.


## Setup

### Install the Google GenAI SDK

Install the Google GenAI SDK from [npm](https://www.npmjs.com/package/@google/genai). 

```bash
$ npm install @google/genai
```

### Setup your API key

You can [create](https://aistudio.google.com/app/apikey) your API key using Google AI Studio with a single click.

Remember to treat your API key like a password. Don't accidentally save it in a notebook or source file you later commit to GitHub. In this notebook we will be storing the API key in a `.env` file. You can also set it as an environment variable or use a secret manager. 

Here's how to set it up in a `.env` file:

```bash
$ touch .env
$ echo "GEMINI_API_KEY=<YOUR_API_KEY>" >> .env
```

:::{.callout-tip}

Another option is to set the API key as an environment variable. You can do this in your terminal with the following command:

```bash
$ export GEMINI_API_KEY="<YOUR_API_KEY>"
```
:::

### Load the API key

To load the API key from the `.env` file, we will use the `dotenv` package. This package loads environment variables from a `.env` file into `process.env`. 

```bash
$ npm install dotenv
```

Then, we can load the API key in our code:


In [2]:
const dotenv = require("dotenv") as typeof import("dotenv");

dotenv.config({
  path: "../.env",
});

const GEMINI_API_KEY = process.env.GEMINI_API_KEY ?? "";
if (!GEMINI_API_KEY) {
  throw new Error("GEMINI_API_KEY is not set in the environment variables");
}
console.log("GEMINI_API_KEY is set in the environment variables");


GEMINI_API_KEY is set in the environment variables


:::{.callout-note}
In our particular case the `.env` is is one directory up from the notebook, hence we need to use `../` to go up one directory. If the `.env` file is in the same directory as the notebook, you can omit it altogether. 

```
│
├── .env
└── quickstarts
    └── Entity_Extraction.ipynb
```
:::


### Initialize SDK Client

With the new SDK, now you only need to initialize a client with you API key (or OAuth if using [Vertex AI](https://cloud.google.com/vertex-ai)). The model is now set in each call.


In [3]:
const google = require("@google/genai") as typeof import("@google/genai");

const ai = new google.GoogleGenAI({ apiKey: GEMINI_API_KEY });


### Select a model

Now select the model you want to use in this guide, either by selecting one in the list or writing it down. Keep in mind that some models, like the 2.5 ones are thinking models and thus take slightly more time to respond (cf. [thinking notebook](../quickstarts/Get_started_thinking.ipynb) for more details and in particular learn how to switch the thiking off).


In [4]:
const tslab = require("tslab") as typeof import("tslab");

const MODEL_ID = "gemini-2.5-flash-preview-05-20";


## Examples


### Extracting few entities at once

This block of text is about possible ways to travel from the airport to the Colosseum.

Let's extract all street names and proposed forms of transportation from it.


In [5]:
const directions = `
  To reach the Colosseum from Rome's Fiumicino Airport (FCO),
  your options are diverse. Take the Leonardo Express train from FCO
  to Termini Station, then hop on metro line A towards Battistini and
  alight at Colosseo station.
  Alternatively, hop on a direct bus, like the Terravision shuttle, from
  FCO to Termini, then walk a short distance to the Colosseum on
  Via dei Fori Imperiali.
  If you prefer a taxi, simply hail one at the airport and ask to be taken
  to the Colosseum. The taxi will likely take you through Via del Corso and
  Via dei Fori Imperiali.
  A private transfer service offers a direct ride from FCO to the Colosseum,
  bypassing the hustle of public transport.
  If you're feeling adventurous, consider taking the train from
  FCO to Ostiense station, then walking through the charming
  Trastevere neighborhood, crossing Ponte Palatino to reach the Colosseum,
  passing by the Tiber River and Via della Lungara.
  Remember to validate your tickets on the metro and buses,
  and be mindful of pickpockets, especially in crowded areas.
  No matter which route you choose, you're sure to be awed by the
  grandeur of the Colosseum.
`;


You will use Gemini 2.5 Flash model for fast responses.


In [8]:
const directionsPrompt = `
  From the given text, extract the following entities and return a list of them.
  Entities to extract: street name, form of transport.
  Text: ${directions}
  Street = []
  Transport = []
`;

const response = await ai.models.generateContent({
  model: MODEL_ID,
  contents: directionsPrompt,
});
tslab.display.markdown(response.text ?? "");


Street = ['Via dei Fori Imperiali', 'Via del Corso', 'Ponte Palatino', 'Via della Lungara']
Transport = ['Leonardo Express train', 'metro', 'bus', 'Terravision shuttle', 'taxi', 'private transfer service', 'public transport']

You can modify the form of the answer for your extracted entities even more:


In [9]:
const directionsListPrompt = `
  From the given text, extract the following entities and
  return a list of them.
  Entities to extract: street name, form of transport.
  Text: ${directions}
  Return your answer as two lists:
  Street = [street names]
  Transport = [forms of transport]
`;

const responseList = await ai.models.generateContent({
  model: MODEL_ID,
  contents: directionsListPrompt,
});
tslab.display.markdown(responseList.text ?? "");


Street = ["Via dei Fori Imperiali", "Via del Corso", "Via della Lungara"]
Transport = ["Leonardo Express train", "metro", "bus", "Terravision shuttle", "taxi", "private transfer service", "train"]

### Numbers

Try entity extraction of phone numbers


In [10]:
const customerServiceEmail = `
  Hello,
  Thank you for reaching out to our customer support team regarding your
  recent purchase of our premium subscription service.
  Your activation code has been sent to +87 668 098 344
  Additionally, if you require immediate assistance, feel free to contact us
  directly at +1 (800) 555-1234.
  Our team is available Monday through Friday from 9:00 AM to 5:00 PM PST.
  For after-hours support, please call our
  dedicated emergency line at +87 455 555 678.
  Thanks for your business and look forward to resolving any issues
  you may encounter promptly.
  Thank you.
`;


In [11]:
const phonePrompt = `
  From the given text, extract the following entities and return a list of them.
  Entities to extract: phone numbers.
  Text: ${customerServiceEmail}
  Return your answer in a list:
`;

const phoneResponse = await ai.models.generateContent({
  model: MODEL_ID,
  contents: phonePrompt,
});
tslab.display.markdown(phoneResponse.text ?? "");


```json
[
  "+87 668 098 344",
  "+1 (800) 555-1234",
  "+87 455 555 678"
]
```

### URLs

Try entity extraction of URLs and get response as a clickable link.


In [12]:
const urlsText = `
  Gemini API billing FAQs

  This page provides answers to frequently asked questions about billing
  for the Gemini API. For pricing information, see the pricing page
  https://ai.google.dev/pricing.
  For legal terms, see the terms of service
  https://ai.google.dev/gemini-api/terms#paid-services.

  What am I billed for?
  Gemini API pricing is based on total token count, with different prices
  for input tokens and output tokens. For pricing information,
  see the pricing page https://ai.google.dev/pricing.

  Where can I view my quota?
  You can view your quota and system limits in the Google Cloud console
  https://console.cloud.google.com/apis/api/generativelanguage.googleapis.com/quotas.

  Is GetTokens billed?
  Requests to the GetTokens API are not billed,
  and they don't count against inference quota.
`;


In [13]:
const urlPrompt = `
  From the given text, extract the following entities and return a list of them.
  Entities to extract: URLs.
  Text: {url_text}
  Do not duplicate entities.
  Return your answer in a markdown format:
`;

const urlResponse = await ai.models.generateContent({
  model: MODEL_ID,
  contents: urlPrompt.replace("{url_text}", urlsText),
});
tslab.display.markdown(urlResponse.text ?? "");


```markdown
- https://ai.google.dev/pricing
- https://ai.google.dev/gemini-api/terms#paid-services
- https://console.cloud.google.com/apis/api/generativelanguage.googleapis.com/quotas
```